# 04 Explainable AI (Phase 6, Gate 6)

Thin notebook over `src/explain.py`. The heavy work (deletion curves, parameter randomisation, SHAP, case studies) runs through `python -m src.explain ...`; this notebook renders what that command wrote under `XAI_DIR` and demonstrates the two local explainers live on a couple of test utterances.

In [ ]:
# parameters: the smoke run sets RUNS_ROOT = 'data/processed/smoke_runs' and the smoke manifest dir
RUNS_ROOT = 'runs'
MANIFEST_DIR = None
FEATURE_ROOT = None
SEED = 13
FUSION = 'fusion_gated'
TEXT_RUN = 'text_roberta'
AUDIO_RUN = 'audio_wavlm_ft'
XAI_DIR = None  # None -> reports/xai (or data/processed/smoke_reports/xai for smoke runs)
N_LIVE = 2  # utterances for the live demo cells

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import Image, Markdown, display
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)
SMOKE = RUNS_ROOT != 'runs'
import numpy as np, torch
from src import explain as xp
from src.evaluate import test_frame
from src.common import EMOTIONS, device_name, load_yaml
XAI_DIR = Path(XAI_DIR) if XAI_DIR else (ROOT / ('data/processed/smoke_reports/xai' if SMOKE else 'reports/xai'))
runs_root = ROOT / RUNS_ROOT
device = device_name()
print(XAI_DIR, device)

## Gate 6 table and faithfulness curves (X1-X3), written by `src.explain`

In [ ]:
for name in ['gate6_xai.md']:
    p = XAI_DIR / name
    display(Markdown(p.read_text()) if p.exists() else Markdown(f'_{name} missing: run `python -m src.explain` first_'))
for f in ['x1_text_deletion.png', 'x2_audio_deletion.png']:
    if (XAI_DIR / f).exists():
        display(Image(filename=str(XAI_DIR / f), width=420))
nums = XAI_DIR / 'xai_numbers.json'
if nums.exists():
    n = json.loads(nums.read_text())['numbers']
    print('X3 text:', n.get('x3_text'))
    print('X3 audio:', n.get('x3_audio'))

## Global explanations: WavLM layer weights, aggregated IG tokens, SHAP on eGeMAPS (X5)

In [ ]:
if (XAI_DIR / 'layer_weights.png').exists():
    display(Image(filename=str(XAI_DIR / 'layer_weights.png'), width=480))
if (XAI_DIR / 'global_tokens_ig.csv').exists():
    gt = pd.read_csv(XAI_DIR / 'global_tokens_ig.csv')
    display(gt.groupby('class').head(10))
if (XAI_DIR / 'x5_shap_vs_eda.csv').exists():
    display(pd.read_csv(XAI_DIR / 'x5_shap_vs_eda.csv'))
    display(Image(filename=str(XAI_DIR / 'x5_shap_egemaps.png'), width=560))

## Modality contribution per class (X4)

In [ ]:
if (XAI_DIR / 'modality_contribution.csv').exists():
    c = pd.read_csv(XAI_DIR / 'modality_contribution.csv')
    display(c.groupby('label').agg(n=('utt_id', 'count'), audio_share=('audio_share', 'mean'), gate=('gate', 'mean'), d_audio=('delta_drop_audio', 'mean'), d_text=('delta_drop_text', 'mean')))
    display(Image(filename=str(XAI_DIR / 'x4_modality_contribution.png'), width=700))

## Case studies (X6)

In [ ]:
p = XAI_DIR / 'cases' / 'cases.md'
if p.exists():
    display(Markdown(p.read_text().split('\n## ')[0]))
    for png in sorted((XAI_DIR / 'cases').glob('*.png'))[:3]:
        display(Markdown(f'**{png.stem}**'))
        display(Image(filename=str(png), width=640))
else:
    print('no cases written yet')

## Live demo: Integrated Gradients on the text run

In [ ]:
from transformers import AutoTokenizer
from src.models import load_run, tokenizer_name
fcfg = load_yaml(runs_root / FUSION / str(SEED) / 'config.yaml')
if MANIFEST_DIR: fcfg['data']['manifest_dir'] = MANIFEST_DIR
df = test_frame(fcfg, MANIFEST_DIR, limit=N_LIVE)
trun = runs_root / TEXT_RUN / str(SEED)
tmodel, tcfg = load_run(trun, device)
tok = AutoTokenizer.from_pretrained(tokenizer_name(tcfg))
texp = xp.TextExplainer(tmodel, tok, device, n_steps=20)
for _, r in df.iterrows():
    ex = texp.explain(str(r['text']))
    print(r['utt_id'], 'true', EMOTIONS[int(r['label'])], 'pred', EMOTIONS[ex['target']], f"p={ex['p_target']:.2f}")
    display(pd.DataFrame({'token': [xp.clean_token(t) for t in ex['tokens']], 'IG': ex['attr'].round(4)}).T)
    from IPython.display import HTML
    display(HTML(xp.heatmap_html(ex['tokens'], ex['attr'], r['utt_id'])))
del tmodel, texp; xp.free()

## Live demo: 250 ms occlusion saliency on the audio run (finetune mode only)

In [ ]:
arun = runs_root / AUDIO_RUN / str(SEED)
amodel, acfg = load_run(arun, device)
print('layer weights', amodel.layer_weights().cpu().numpy().round(3))
if getattr(amodel, 'mode', 'cached') == 'finetune':
    occ = xp.AudioOccluder(amodel, device, 4)
    import matplotlib.pyplot as plt
    for _, r in df.iterrows():
        wav = xp.load_wav(r['wav_path'], float(fcfg['train'].get('max_audio_s', 10)))
        ex = occ.explain(wav)
        fig, (a1, a2) = plt.subplots(2, 1, figsize=(9, 4), sharex=True, height_ratios=[2, 1])
        xp.spectrogram_saliency(a1, a2, wav, ex, f"{r['utt_id']} true={EMOTIONS[int(r['label'])]} pred={EMOTIONS[ex['target']]} p={ex['p_target']:.2f}")
        plt.show()
else:
    print('cached-mode audio run: occlusion needs the finetune path')
del amodel; xp.free()